# Protecting Customers' Personal Data (Linear Algebra)

## Project Description

An insurance company wants to protect the personal data of its clients. The data must be transformed (encrypted) so that it is hard to recover the original information, while the quality of machine learning models does not get worse.

**Tasks:**
- Load and study the data.
- Answer the question and justify the solution: the features are multiplied by an invertible matrix. Will the quality of linear regression change? (The model can be retrained.)
    - a. It will change. Give examples of matrices.
    - b. It will not change. State how the parameters of the linear regression on the original and on the transformed data are related.
- Propose an algorithm for transforming the data. Justify why the quality of linear regression does not change.
- Implement the algorithm with matrix operations. Check that the quality of the linear regression (R2 metric) is the same before and after the transformation.

## 1. Data Loading

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv('insurance.csv')

# rename the columns (the original column names in the file are in Russian)
data = data.rename(columns={'Пол': 'gender',
                            'Возраст': 'age',
                            'Зарплата': 'salary',
                            'Члены семьи': 'family_members',
                            'Страховые выплаты': 'insurance_payments'})

In [3]:
data

,gender,age,salary,family_members,insurance_payments
0,1,41.0,49600.0,1,0
1,0,46.0,38000.0,1,1
2,0,29.0,21000.0,0,0
3,0,21.0,41700.0,2,0
4,1,28.0,26100.0,0,0
...,...,...,...,...,...
4995,0,28.0,35700.0,2,0
4996,0,34.0,52400.0,1,0
4997,0,20.0,33900.0,2,0
4998,1,22.0,32700.0,3,0


In [4]:
data.shape

(5000, 5)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   gender              5000 non-null   int64  
 1   age                 5000 non-null   float64
 2   salary              5000 non-null   float64
 3   family_members      5000 non-null   int64  
 4   insurance_payments  5000 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 195.4 KB


## 2. Matrix Multiplication

Notation:

- $X$ - feature matrix (the zeroth column consists of ones)

- $y$ - target vector

- $P$ - matrix that the features are multiplied by

- $w$ - vector of linear regression weights (the zeroth element is the intercept)

Predictions:

$$
a = Xw
$$

Training objective:

$$
w = \arg\min_w MSE(Xw, y)
$$

Training formula:

$$
w = (X^T X)^{-1} X^T y
$$

**Answer:** The quality of linear regression does not change (the weights change, the predictions do not), provided that the matrix $P$ is invertible.

**Justification:** Multiply the features by the matrix $P$. The weights trained on the transformed features $XP$ are

$$
w_P = ((XP)^T XP)^{-1} (XP)^T y
$$

Apply $(AB)^T = B^T A^T$ and $(AB)^{-1} = B^{-1} A^{-1}$ (valid because $P$ is square and invertible):

$$
w_P = (P^T X^T X P)^{-1} P^T X^T y = P^{-1} (X^T X)^{-1} (P^T)^{-1} P^T X^T y = P^{-1} (X^T X)^{-1} X^T y = P^{-1} w
$$

Substitute the result into the predictions:

$$
a_P = X P w_P = X P P^{-1} w = X w = a
$$

The predictions are identical, so the MSE and the R2 metric are identical.

So the models trained on the original and on the encrypted data give the same results.

# 3. Transformation Algorithm
Task: encrypt the data so that the quality of machine learning models does not get worse.

**Algorithm**

1. Generate a random square matrix whose size equals the number of features. Check that it is invertible with `np.linalg.inv()` (it raises an error for a singular matrix).
2. Multiply the feature matrix by this matrix. The result is the encrypted data.
3. Train the linear regression on the original and on the encrypted features and compare the R2 metric.

**Justification**

From the formulas of the previous section, the quality of linear regression does not change after multiplying the features by an invertible matrix. The original data can be recovered by multiplying the encrypted features by the inverse matrix.

In [6]:
# Create a random matrix (fixed seed for reproducibility)
np.random.seed(42)
a = np.random.rand(4, 4)
a

array([[0.37454012, 0.95071431, 0.73199394, 0.59865848],
       [0.15601864, 0.15599452, 0.05808361, 0.86617615],
       [0.60111501, 0.70807258, 0.02058449, 0.96990985],
       [0.83244264, 0.21233911, 0.18182497, 0.18340451]])

In [7]:
# Check that the matrix is invertible (np.linalg.inv raises an error otherwise)
a1 = np.linalg.inv(a)
a1

array([[-0.32076901, -0.12766508,  0.06141427,  1.32518674],
       [ 0.35151041, -1.88500014,  1.65560045, -1.0003883 ],
       [ 1.14080312,  1.3467702 , -2.0407373 ,  0.70794071],
       [-0.08202687,  1.42666425, -0.17238177, -0.10600441]])

In [8]:
# Train the first model on the original (not encrypted) data
features = data.drop('insurance_payments', axis=1)
target = data['insurance_payments']

class LinearRegression:
    def fit(self, train_features, train_target):
        # add a column of ones for the intercept
        X = np.concatenate((np.ones((train_features.shape[0], 1)), train_features), axis=1)
        y = train_target
        # w = (X^T X)^-1 X^T y
        w = (np.linalg.inv(X.T.dot(X)).dot(X.T)).dot(y)
        self.w = w[1:]
        self.w0 = w[0]

    def predict(self, test_features):
        return test_features.dot(self.w) + self.w0
    
model = LinearRegression()
model.fit(features, target)
predictions = model.predict(features)
print(r2_score(target, predictions))

0.42494550286668


In [9]:
features_vec = features.values
features_vec

array([[1.00e+00, 4.10e+01, 4.96e+04, 1.00e+00],
       [0.00e+00, 4.60e+01, 3.80e+04, 1.00e+00],
       [0.00e+00, 2.90e+01, 2.10e+04, 0.00e+00],
       ...,
       [0.00e+00, 2.00e+01, 3.39e+04, 2.00e+00],
       [1.00e+00, 2.20e+01, 3.27e+04, 3.00e+00],
       [1.00e+00, 2.80e+01, 4.06e+04, 1.00e+00]])

In [10]:
# Multiply the features by the random matrix to encrypt them
features_encrypt = features_vec@a
features_encrypt

array([[29822.90832948, 35127.95868743,  1024.28616408, 48143.82395221],
       [22850.37974634, 26914.1460433 ,   785.06445437, 36896.60188937],
       [12627.93978718, 14874.04797481,   433.95880496, 20393.22600363],
       ...,
       [20382.58415619, 24007.20495591,   699.33967881, 32897.63432023],
       [19662.76516213, 23158.99290502,   675.66827178, 31736.25691292],
       [24410.84498147, 28753.27755851,   838.27062846, 39403.37499285]])

## 4. Algorithm Check

In [11]:
# Retrain the model on the encrypted data and look at the result
model = LinearRegression()
model.fit(features_encrypt, target)
predictions_encrypt = model.predict(features_encrypt)
print(r2_score(target, predictions_encrypt))

0.4249455028666642


## Conclusion

- The data is loaded and checked: 5000 clients, 4 features (gender, age, salary, number of family members) and the target (number of insurance payments). There are no missing values; all columns are integers except age and salary, which are stored as floats.
- It is proved that multiplying the features by an invertible matrix $P$ does not change the quality of linear regression: the weights become $w_P = P^{-1} w$ and the predictions stay the same.
- The algorithm is implemented: the features are multiplied by a random invertible 4x4 matrix. The R2 of the linear regression is the same before and after the encryption (0.425).
- The client data becomes unreadable without the key, and it can be recovered by multiplying the encrypted features by the inverse matrix.